In [0]:
import os

# Vérifie que le fichier est bien uploadé
dbutils.fs.ls("dbfs:/Volumes/kitchen-proline-data/raw/data-files/")

In [0]:
import zipfile

# Dézipper le fichier
with zipfile.ZipFile("/Volumes/kitchen-proline-data/raw/data-files/Données.zip", "r") as zip_ref:
    zip_ref.extractall("/Volumes/kitchen-proline-data/raw/data-files")

# Vérifier le contenu
dbutils.fs.ls("/Volumes/kitchen-proline-data/raw/data-files")

In [0]:
# Voir ce qu'il y a dans le dossier Données
dbutils.fs.ls("/Volumes/kitchen-proline-data/raw/data-files/Données/")

In [0]:
# Voir les fichiers Refs
print("=== REFS ===")
dbutils.fs.ls("/Volumes/kitchen-proline-data/raw/data-files/Données/Refs/")

## Exploration des fichiers de référence
Examinons la structure de quelques fichiers TSV pour comprendre leur format avant de créer les dimensions.

In [0]:
# Examinons d'abord le fichier Pays (petit fichier)
df_pays_sample = spark.read.csv(
    "/Volumes/kitchen-proline-data/raw/data-files/Données/Refs/Pays.txt",
    sep="\t",
    header=True,
    inferSchema=True
)

print("Schéma Pays:")
df_pays_sample.printSchema()
print("\nÉchantillon de données:")
display(df_pays_sample.limit(5))

In [0]:
# Fichier Magasin
df_magasin_sample = spark.read.csv(
    "/Volumes/kitchen-proline-data/raw/data-files/Données/Refs/Magasin.txt",
    sep="\t",
    header=True,
    inferSchema=True
)

print("Schéma Magasin:")
df_magasin_sample.printSchema()
print("\nÉchantillon de données:")
display(df_magasin_sample.limit(5))

In [0]:
# Fichier Client
df_client_sample = spark.read.csv(
    "/Volumes/kitchen-proline-data/raw/data-files/Données/Refs/Client.txt",
    sep="\t",
    header=True,
    inferSchema=True
)

print("Schéma Client:")
df_client_sample.printSchema()
print("\nÉchantillon de données:")
display(df_client_sample.limit(5))

## Création des dimensions simples
Chargement des fichiers TSV et création des tables de dimensions dans le schéma `sales`

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Définition des chemins
BASE_PATH = "/Volumes/kitchen-proline-data/raw/data-files/Données/Refs/"
CATALOG = "kitchen-proline-data"
SCHEMA = "sales"

# Fonction générique pour charger les fichiers TSV
def load_tsv_file(file_name):
    """
    Charge un fichier TSV depuis le répertoire Refs
    """
    return spark.read.csv(
        f"{BASE_PATH}{file_name}",
        sep="\t",
        header=True,
        inferSchema=True
    )

# Fonction pour sauvegarder une dimension
def save_dimension(df, table_name, mode="overwrite"):
    """
    Sauvegarde un DataFrame comme table de dimension dans Unity Catalog
    """
    # Ajout des backticks pour les noms avec caractères spéciaux
    full_table_name = f"`{CATALOG}`.`{SCHEMA}`.{table_name}"
    # Option overwriteSchema pour permettre l'ajout de colonnes ID
    df.write.mode(mode).option("overwriteSchema", "true").saveAsTable(full_table_name)
    print(f"Table {full_table_name} créée avec succès ({df.count()} lignes)")
    return full_table_name

print("✓ Configuration et fonctions chargées")

In [0]:
# ===== DimPays =====
print("Création de DimPays...")
df_pays = load_tsv_file("Pays.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("CodePays")

df_dim_pays = df_pays.select(
    F.row_number().over(window_spec).alias("id_pays"),
    F.col("CodePays").alias("code_pays"),
    F.col("LibPays").alias("libelle_pays"),
    F.col("DrapeauImg").alias("drapeau_url")
)

# Sauvegarde
save_dimension(df_dim_pays, "DimPays")
display(df_dim_pays.limit(5))

In [0]:
# ===== DimMagasin =====
print("Création de DimMagasin...")
df_magasin = load_tsv_file("Magasin.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("CodeMagasin")

df_dim_magasin = df_magasin.select(
    F.row_number().over(window_spec).alias("id_magasin"),
    F.col("CodeMagasin").alias("code_magasin"),
    F.col("LibMagasin").alias("libelle_magasin"),
    F.trim(F.col("Pays")).alias("code_pays")
)

# Sauvegarde
save_dimension(df_dim_magasin, "DimMagasin")
display(df_dim_magasin)

In [0]:
# ===== DimMarque =====
print("Création de DimMarque...")
df_marque = load_tsv_file("Marque.txt")

# Affichage du schéma pour comprendre la structure
print("\nSchéma du fichier Marque:")
df_marque.printSchema()

print("\nÉchantillon de données:")
display(df_marque.limit(5))

In [0]:
# Explorons les autres fichiers pour comprendre leur structure

print("=== FOURNISSEUR ===")
df_fournisseur = load_tsv_file("Fournisseur.txt")
df_fournisseur.printSchema()
display(df_fournisseur.limit(3))

print("\n=== FACADE ===")
df_facade = load_tsv_file("Facade.txt")
df_facade.printSchema()
display(df_facade.limit(3))

print("\n=== FAMILLE PRODUIT ===")
df_famille = load_tsv_file("FamilleProduit.txt")
df_famille.printSchema()
display(df_famille.limit(3))

In [0]:
# ===== DimMarque =====
print("Création de DimMarque...")
df_marque = load_tsv_file("Marque.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("CodeMarque")

df_dim_marque = df_marque.select(
    F.row_number().over(window_spec).alias("id_marque"),
    F.col("CodeMarque").alias("code_marque"),
    F.when(F.col("LibMarque") == "NULL", None).otherwise(F.col("LibMarque")).alias("libelle_marque")
)

# Sauvegarde
save_dimension(df_dim_marque, "DimMarque")
display(df_dim_marque.limit(10))

In [0]:
# ===== DimFournisseur =====
print("Création de DimFournisseur...")
df_fournisseur = load_tsv_file("Fournisseur.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("CodeFournisseur")

df_dim_fournisseur = df_fournisseur.select(
    F.row_number().over(window_spec).alias("id_fournisseur"),
    F.col("CodeFournisseur").alias("code_fournisseur"),
    F.when(F.col("LibFournisseur") == "NULL", None).otherwise(F.col("LibFournisseur")).alias("libelle_fournisseur")
)

# Sauvegarde
save_dimension(df_dim_fournisseur, "DimFournisseur")
display(df_dim_fournisseur.limit(10))

In [0]:
# ===== DimFacade =====
print("Création de DimFacade...")
df_facade = load_tsv_file("Facade.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("Code")

df_dim_facade = df_facade.select(
    F.row_number().over(window_spec).alias("id_facade"),
    F.col("Code").alias("code_facade"),
    F.col("Libelle").alias("libelle_facade"),
    F.col("Marque").alias("code_marque")
)

# Sauvegarde
save_dimension(df_dim_facade, "DimFacade")
display(df_dim_facade.limit(10))

In [0]:
# ===== DimFamilleProduit =====
print("Création de DimFamilleProduit...")
df_famille = load_tsv_file("FamilleProduit.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("CodeFamille")

df_dim_famille = df_famille.select(
    F.row_number().over(window_spec).alias("id_famille"),
    F.col("CodeFamille").alias("code_famille"),
    F.col("LibFamille").alias("libelle_famille"),
    F.col("NiveauFamille").alias("niveau_famille")
)

# Sauvegarde
save_dimension(df_dim_famille, "DimFamilleProduit")
display(df_dim_famille.limit(10))

In [0]:
# Explorons les fichiers Vendeur et Modele

print("=== VENDEUR ===")
df_vendeur = load_tsv_file("Vendeur.txt")
df_vendeur.printSchema()
display(df_vendeur.limit(5))

print("\n=== MODELE ===")
df_modele = load_tsv_file("Model.txt")
df_modele.printSchema()
display(df_modele.limit(5))

In [0]:
# ===== DimVendeur =====
print("Création de DimVendeur...")
df_vendeur = load_tsv_file("Vendeur.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("CodeUtilisateur")

df_dim_vendeur = df_vendeur.select(
    F.row_number().over(window_spec).alias("id_vendeur"),
    F.col("CodeUtilisateur").alias("code_vendeur"),
    F.col("NomPrenom").alias("nom_prenom_vendeur"),
    F.col("Magasin").alias("code_magasin")
)

# Sauvegarde
save_dimension(df_dim_vendeur, "DimVendeur")
display(df_dim_vendeur.limit(10))

In [0]:
# ===== DimModele =====
print("Création de DimModele...")
df_modele = load_tsv_file("Model.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("CodeModel")

df_dim_modele = df_modele.select(
    F.row_number().over(window_spec).alias("id_modele"),
    F.col("CodeModel").alias("code_modele"),
    F.col("TypeModel").alias("type_modele"),
    F.col("UniteMesure").alias("unite_mesure"),
    F.when(F.col("Famille") == "NULL", None).otherwise(F.col("Famille")).alias("code_famille"),
    F.when(F.col("Longueur") == "NULL", None).otherwise(F.col("Longueur")).alias("longueur"),
    F.when(F.col("Hauteur") == "NULL", None).otherwise(F.col("Hauteur")).alias("hauteur"),
    F.when(F.col("Profondeur") == "NULL", None).otherwise(F.col("Profondeur")).alias("profondeur"),
    F.when(F.col("Poids") == "NULL", None).otherwise(F.col("Poids")).alias("poids"),
    F.trim(F.col("Pays")).alias("code_pays"),
    F.col("Marque").alias("code_marque"),
    F.col("Fournisseur").alias("code_fournisseur")
)

# Sauvegarde
save_dimension(df_dim_modele, "DimModele")
display(df_dim_modele.limit(10))

In [0]:
# Explorons le fichier Produit
print("=== PRODUIT ===")
df_produit = load_tsv_file("Produit.txt")
df_produit.printSchema()
print(f"\nNombre de produits : {df_produit.count()}")
display(df_produit.limit(5))

In [0]:
# ===== DimProduit =====
print("Création de DimProduit...")
df_produit = load_tsv_file("Produit.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("CodeProduit")

df_dim_produit = df_produit.select(
    F.row_number().over(window_spec).alias("id_produit"),
    F.col("CodeProduit").alias("code_produit"),
    F.col("LibProduit").alias("libelle_produit"),
    F.col("DescProduit").alias("description_produit"),
    F.when(F.col("TypeProduit") == "NULL", None).otherwise(F.col("TypeProduit")).alias("type_produit"),
    F.when(F.col("UniteMesure") == "NULL", None).otherwise(F.col("UniteMesure")).alias("unite_mesure"),
    F.col("CodeFamille").alias("code_famille"),
    F.when(F.col("Longueur") == "NULL", None).otherwise(F.col("Longueur")).alias("longueur"),
    F.when(F.col("Hauteur") == "NULL", None).otherwise(F.col("Hauteur")).alias("hauteur"),
    F.when(F.col("Profondeur") == "NULL", None).otherwise(F.col("Profondeur")).alias("profondeur"),
    F.when(F.col("Poids") == "NULL", None).otherwise(F.col("Poids")).alias("poids"),
    F.when(F.col("Volume") == "NULL", None).otherwise(F.col("Volume")).alias("volume"),
    F.when(F.col("CodeEAN") == "NULL", None).otherwise(F.col("CodeEAN")).alias("code_ean"),
    F.col("EstReference").alias("est_reference"),
    # Nettoyer les prix (remplacer virgule par point pour conversion numérique)
    F.when(F.col("PrixAchat") == "NULL", None).otherwise(F.regexp_replace(F.col("PrixAchat"), ",", ".")).cast("decimal(10,2)").alias("prix_achat"),
    F.when(F.col("PrixVente") == "NULL", None).otherwise(F.regexp_replace(F.col("PrixVente"), ",", ".")).cast("decimal(10,2)").alias("prix_vente"),
    F.col("CodeMagasin").alias("code_magasin"),
    F.trim(F.col("CodePays")).alias("code_pays"),
    F.col("codeMarque").alias("code_marque"),
    F.col("CodeFournisseur").alias("code_fournisseur")
)

# Sauvegarde
save_dimension(df_dim_produit, "DimProduit")
print(f"\nÉchantillon de prix:")
display(df_dim_produit.select("id_produit", "code_produit", "libelle_produit", "prix_achat", "prix_vente").limit(10))

In [0]:
# ===== DimClient =====
print("Création de DimClient...")
df_client = load_tsv_file("Client.txt")

# Nettoyage et transformation avec surrogate key
from pyspark.sql.window import Window
window_spec = Window.orderBy("CodeClient")

df_dim_client = df_client.select(
    F.row_number().over(window_spec).alias("id_client"),
    F.col("CodeClient").alias("code_client"),
    F.col("Civilite").alias("civilite"),
    F.col("NomPrenom1").alias("nom_prenom_1"),
    F.col("NomPrenom2").alias("nom_prenom_2"),
    F.col("Telephone").alias("telephone"),
    F.col("Telephone1").alias("telephone_1"),
    F.col("Telephone2").alias("telephone_2"),
    F.col("Email").alias("email"),
    F.col("Adresse").alias("adresse"),
    F.col("CodePostal").alias("code_postal"),
    F.col("Ville").alias("ville"),
    F.trim(F.col("Pays")).alias("code_pays"),
    F.col("CodeMagasin").alias("code_magasin")
)

# Sauvegarde
save_dimension(df_dim_client, "DimClient")
display(df_dim_client.limit(10))

## 🎉 Récapitulatif des dimensions créées
Toutes les tables de dimensions ont été créées avec succès dans le schéma `kitchen-proline-data.sales` !

In [0]:
%sql
-- Affichage de toutes les tables du schéma sales
SHOW TABLES IN `kitchen-proline-data`.`sales`

In [0]:
# Affichage des statistiques pour chaque dimension
from pyspark.sql import functions as F

dimensions = [
    ("DimPays", "Pays"),
    ("DimMagasin", "Magasins"),
    ("DimMarque", "Marques"),
    ("DimFournisseur", "Fournisseurs"),
    ("DimFacade", "Façades"),
    ("DimFamilleProduit", "Familles de produits"),
    ("DimVendeur", "Vendeurs"),
    ("DimModele", "Modèles"),
    ("DimProduit", "Produits"),
    ("DimClient", "Clients")
]

print("="*60)
print("STATISTIQUES DES DIMENSIONS")
print("="*60)

for table_name, description in dimensions:
    full_table = f"`kitchen-proline-data`.`sales`.{table_name}"
    count = spark.table(full_table).count()
    print(f"{description:30} : {count:6} enregistrements ({table_name})")

print("="*60)
print("✓ Toutes les dimensions ont été créées avec succès !")
print("="*60)